<a href="https://colab.research.google.com/github/Anorwed/vkr/blob/main/picoqwen3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Для запуска нажмите кнопку слева от "Показать код"
import os
import re
import time
import json
import subprocess
import sys
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

def run_installation(vk_token, community_url):
    # --- Валидация ---
    vk_token = vk_token.strip()
    community_url = community_url.strip()

    if not vk_token:
        print("❌ Ошибка: Токен сообщества VK не может быть пустым!")
        return
    if not community_url:
        print("❌ Ошибка: Ссылка на сообщество VK не может быть пустой!")
        return

    # Извлечение group_id
    group_id = None
    m = re.search(r'(?:vk\.com|vk\.ru)/(?:club|public)(\d+)', community_url)
    if not m:
        m = re.search(r'^(?:club|public)(\d+)$', community_url)
    if m:
        group_id = int(m.group(1))
    else:
        print("❌ Ошибка: Не удалось извлечь ID сообщества.")
        print("   Поддерживаемые форматы: https://vk.com/club123456789 или public123456789")
        return

    print(f"=== PicoClaw + Ollama + VK (оптимизировано) ===")
    print(f"🆔 VK Group ID: {group_id}")
    print(f"🔧 Подготовка системы...")
    subprocess.run('apt-get update -qq && apt-get install -y -qq zstd pciutils net-tools',
                   shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    print(f"🦙 Установка Ollama...")
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                   shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    print(f"🚀 Запуск Ollama...")
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    subprocess.Popen(['nohup', 'ollama', 'serve'], stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
    time.sleep(12)

    # Проверка Ollama
    print(f"🔍 Проверка Ollama...")
    for i in range(6):
        try:
            import urllib.request
            with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3) as r:
                if r.status == 200:
                    print("   ✅ Ollama отвечает")
                    break
        except Exception:
            time.sleep(2)
    else:
        print("   ⚠️ Ollama не отвечает, продолжаем...")

    # Загрузка модели (4B вместо 9B)
    print(f"📥 Загрузка qwen3.5:4b...")
    subprocess.run(['ollama', 'pull', 'qwen3.5:4b'])

    # Проверка загрузки на GPU
    print(f"🔍 Проверка модели...")
    time.sleep(2)
    ps = subprocess.check_output(['ollama', 'ps'], text=True)
    print(ps)
    if "qwen3.5:4b" not in ps:
        print("   ⚠️ Модель не найдена в ollama ps")

    # Загрузка PicoClaw
    print(f"📥 Загрузка PicoClaw...")
    subprocess.run('wget -q https://github.com/sipeed/picoclaw/releases/latest/download/picoclaw_Linux_x86_64.tar.gz',
                   shell=True, check=True)
    subprocess.run('tar -xzf picoclaw_Linux_x86_64.tar.gz', shell=True, check=True)
    subprocess.run('chmod +x picoclaw', shell=True)

    # Проверка бинарника
    print(f"🔍 Проверка PicoClaw...")
    try:
        ver = subprocess.check_output(['./picoclaw', 'version'], text=True, stderr=subprocess.STDOUT).strip()
        print(f"   ✅ Версия: {ver}")
    except Exception:
        ver = subprocess.check_output(['./picoclaw', '--version'], text=True, stderr=subprocess.STDOUT).strip()
        print(f"   ✅ Версия: {ver}")

    # Директории
    os.makedirs("/content/workspace", exist_ok=True)
    os.makedirs("/root/.picoclaw", exist_ok=True)

    # === ОПТИМИЗИРОВАННАЯ КОНФИГУРАЦИЯ ===
    config_path = Path.home() / ".picoclaw" / "config.json"
    config_data = {
        "version": 3,
        "agents": {
            "defaults": {
                "workspace": "/content/workspace",
                "model_name": "qwen-local",
                "max_tokens": 2048,           # ← уменьшено для скорости
                "temperature": 0.7,
                "max_tool_iterations": 10,
                "restrict_to_workspace": True
            }
        },
        "model_list": [
            {
                "model_name": "qwen-local",
                "model": "ollama/qwen3.5:4b"  # ← 4B вместо 9B
            }
        ],
        "providers": {
            "ollama": {
                "api_key": "ollama",
                "api_base": "http://localhost:11434/v1"
            }
        },
        "channel_list": {
            "vk": {
                "enabled": True,
                "type": "vk",
                "allow_from": [],
                "typing": {"enabled": False},
                "settings": {
                    "token": vk_token,
                    "group_id": group_id
                }
            }
        },
        # ← Отключены тяжёлые инструменты, оставлены только базовые
        # Это сокращает системный промпт с ~3300 до ~1200-1500 токенов
        "tools": {
            "append_file":   {"enabled": False},
            "cron":          {"enabled": False},
            "edit_file":     {"enabled": False},
            "exec":          {"enabled": True,  "timeout_seconds": 60},
            "find_skills":   {"enabled": False},
            "install_skill": {"enabled": False},
            "list_dir":      {"enabled": True},
            "load_image":    {"enabled": False},
            "message":       {"enabled": True},
            "reaction":      {"enabled": False},
            "read_file":     {"enabled": True,  "mode": "bytes", "max_read_file_size": 65536},
            "send_file":     {"enabled": False},
            "spawn":         {"enabled": False},
            "spawn_status":  {"enabled": False},
            "subagent":      {"enabled": False},
            "web_fetch":     {"enabled": True},
            "write_file":    {"enabled": True}
        },
        "gateway": {
            "host": "0.0.0.0",
            "port": 18790,
            "log_level": "debug"
        }
    }

    with open(config_path, "w") as f:
        json.dump(config_data, f, indent=4)

    # Диагностика
    diag = json.loads(json.dumps(config_data))
    diag["channel_list"]["vk"]["settings"]["token"] = vk_token[:10] + "..." if len(vk_token) > 10 else "***"
    print(f"\n📄 Конфиг (диагностика):")
    print(json.dumps(diag, indent=2))

    print(f"\n✅ Конфиг записан: {config_path}")
    print(f"📋 Модель: ollama/qwen3.5:4b (max_tokens=2048)")
    print(f"🛠️  Инструменты: message, read_file, write_file, exec, list_dir, web_fetch")
    print(f"💬 Канал: VK (group_id={group_id})")
    print(f"\n🚀 Запуск PicoClaw Gateway...")
    print(f"   (для остановки прервите ячейку Ctrl+M I)\n")
    print("-" * 50)

    env = os.environ.copy()
    env['PICOCLAW_LOG_LEVEL'] = 'debug'

    process = subprocess.Popen(
        ['./picoclaw', 'gateway'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
        universal_newlines=True
    )

    try:
        for line in iter(process.stdout.readline, ''):
            if not line:
                break
            print(line, end='')
            sys.stdout.flush()
    except KeyboardInterrupt:
        print("\n🛑 Остановка по запросу пользователя...")
    finally:
        process.stdout.close()
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
        print(f"\n⚠️ PicoClaw завершился с кодом: {process.returncode}")

# --- Интерфейс ---
token_input = widgets.Password(
    value='',
    placeholder='vk1.a.xxx... (токен сообщества)',
    description='VK Token:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

url_input = widgets.Text(
    value='',
    placeholder='https://vk.com/club123456789',
    description='Ссылка:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

start_button = widgets.Button(
    description='🚀 Запустить',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='180px')
)

output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        run_installation(token_input.value, url_input.value)

start_button.on_click(on_button_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>🦞 PicoClaw + Ollama + VK (оптимизировано)</h3>"),
    widgets.HTML("<p>Модель: <b>Qwen 3.5 (4B)</b> | Канал: <b>VK</b> | Max tokens: <b>2048</b></p>"),
    widgets.HTML("<p><small>Токен: Управление сообществом → Работа с API → Ключи доступа (разрешение <b>messages</b>)</small></p>"),
    token_input,
    url_input,
    start_button,
    output
]))
